<a href="https://colab.research.google.com/github/HarryCSK3/CodeCrusher/blob/Projects/NumberRecog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [24]:
df_train = pd.read_csv('/content/sample_data/mnist_train_small.csv')
df_test = pd.read_csv('/content/sample_data/mnist_test.csv')

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print(df_train.head())

Train shape: (19999, 785)
Test shape: (9999, 785)
   6  0  0.1  0.2  0.3  0.4  0.5  0.6  0.7  0.8  ...  0.581  0.582  0.583  \
0  5  0    0    0    0    0    0    0    0    0  ...      0      0      0   
1  7  0    0    0    0    0    0    0    0    0  ...      0      0      0   
2  9  0    0    0    0    0    0    0    0    0  ...      0      0      0   
3  5  0    0    0    0    0    0    0    0    0  ...      0      0      0   
4  2  0    0    0    0    0    0    0    0    0  ...      0      0      0   

   0.584  0.585  0.586  0.587  0.588  0.589  0.590  
0      0      0      0      0      0      0      0  
1      0      0      0      0      0      0      0  
2      0      0      0      0      0      0      0  
3      0      0      0      0      0      0      0  
4      0      0      0      0      0      0      0  

[5 rows x 785 columns]


In [27]:
data = np.array(df_train)
np.random.shuffle(data)

# Split: 1000 for dev, rest for training
m = data.shape[0]

data_dev = data[:1000].T
Y_dev = data_dev[0].astype(int)
X_dev = data_dev[1:] / 255.0

data_train = data[1000:].T
Y_train = data_train[0].astype(int)
X_train = data_train[1:] / 255.0

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_dev shape:", X_dev.shape)
print("Y_dev shape:", Y_dev.shape)

X_train shape: (784, 18999)
Y_train shape: (18999,)
X_dev shape: (784, 1000)
Y_dev shape: (1000,)


In [28]:
X_dev

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [37]:
X_dev[:, 0].shape

(784,)

In [30]:
def __init_param():
    W1 = np.random.randn(10, 784) * 0.01  # Small random values
    b1 = np.zeros((10, 1))  # Initialize biases to zero
    W2 = np.random.randn(10, 10) * 0.01
    b2 = np.zeros((10, 1))
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(Z, 0)

def ReLU_deriv(Z):
    return Z > 0  # Returns 1 where Z > 0, else 0

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))  # Numerical stability
    A = exp_Z / np.sum(exp_Z, axis=0, keepdims=True)
    return A

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    one_hot_Y = one_hot(Y)
    m = Y.size
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, axis=1, keepdims=True)  # axis=1, keepdims
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)  # ReLU derivative
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis=1, keepdims=True)  # axis=1, keepdims
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

In [34]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, X_dev, Y_dev, alpha, iterations):
    W1, b1, W2, b2 = __init_param()
    for i in range(iterations):
        # Train on training data
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

        if i % 50 == 0:
            # Training accuracy
            predictions_train = get_predictions(A2)
            train_acc = get_accuracy(predictions_train, Y)

            # Dev accuracy (the real metric!)
            _, _, _, A2_dev = forward_prop(W1, b1, W2, b2, X_dev)
            predictions_dev = get_predictions(A2_dev)
            dev_acc = get_accuracy(predictions_dev, Y_dev)

            print(f"Iteration {i} | Train: {train_acc:.4f} | Dev: {dev_acc:.4f}")

    return W1, b1, W2, b2

In [36]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, X_dev, Y_dev,
                                   alpha=0.5, iterations=500)

Iteration 0 | Train: 0.0939 | Dev: 0.1250
Iteration 50 | Train: 0.7810 | Dev: 0.7690
Iteration 100 | Train: 0.8772 | Dev: 0.8770
Iteration 150 | Train: 0.8951 | Dev: 0.8820
Iteration 200 | Train: 0.9053 | Dev: 0.8980
Iteration 250 | Train: 0.9179 | Dev: 0.9110
Iteration 300 | Train: 0.9180 | Dev: 0.9140
Iteration 350 | Train: 0.9259 | Dev: 0.9180
Iteration 400 | Train: 0.9220 | Dev: 0.9170
Iteration 450 | Train: 0.9323 | Dev: 0.9200
